# Rates visualization
### Import libraries

In [1]:
import os
import sys
import copy
import time
import pickle
import importlib
import numpy as np
import tensorflow as tf
import joblib as jl
import h5py
import pandas as pd

threads = 4

from tqdm.notebook import tqdm

Visualization libraries

In [2]:
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
%matplotlib qt

Global paths

In [3]:
# WORKSPACE
WORKSPACE = os.environ['WORKSPACE_PATH']
# PyCOMET
PyCOMET = WORKSPACE + '/ML/PyCOMET/pycomet/'
if importlib.util.find_spec('pycomet') is None:
    sys.path.append(os.path.abspath(PyCOMET))
# SurQCT
SurQCT = WORKSPACE + '/ML/PyCOMET/pycomet/examples/SurQCT/'
sys.path.append(os.path.abspath(SurQCT))
# Current Working Directory
CWD = os.getcwd()

In [4]:
# Output directory
out_dir = CWD + '/figs/'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [5]:
# # Get colors list
# colors = mpl.rcParams['axes.prop_cycle'].by_key()['color']*10

# # Font parameters
# mpl.rcParams['text.latex.preamble'].join([
#     r'\usepackage{amsmath}',
#     r'\boldmath',
#     r'\usepackage{pifont}',
#     r'\usepackage{tikz}',
#     r'\usepackage[utf8]{inputenc}'
# ])
# mpl.rcParams['font.family'] = 'lmodern'

# # Tick parameters
# mpl.rcParams['xtick.labelsize'] = 40
# mpl.rcParams['ytick.labelsize'] = 40
# mpl.rcParams['xtick.major.size'] = 20
# mpl.rcParams['xtick.major.width'] = 2
# mpl.rcParams['xtick.minor.size'] = 10
# mpl.rcParams['xtick.minor.width'] = 2

# # Tick parameters
# mpl.rcParams['axes.labelsize'] = 40
# mpl.rcParams['axes.titlesize'] = 40
# mpl.rcParams['legend.fontsize'] = 35


### Initialize PyCOMET model
Working case

In [6]:
case_dir = WORKSPACE + '/ML/PyCOMET/run/SurQCT/mimmo/O3_UMN/KInel/case.01/'
sys.path.append(os.path.abspath(case_dir))

In [7]:
os.chdir(case_dir)

Import additional libraries

In [8]:
from pycomet.utilities import utils
from pycomet.apps import functions as fun
# Inputs
import inputs
from inputs.general import flags

Compile model

In [9]:
# Load network architecture
net = fun._net(
    input_mod=inputs
)
# Finalize model
for k in ('train', 'save'):
    flags[k] = 0
flags['load'] = 1
model = fun._model(
    flags=flags,
    input_mod=inputs,
    net=net
)


[PyCOMET]: Initializing 'OpNN' network ...

[PyCOMET]: Building 'OpNN' network ...
Model: "SurQCT_OpNN"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 5)]          0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 5)]          0                                            
__________________________________________________________________________________________________
normalization (Normalization)   (None, 5)            11          input_1[0][0]                    
__________________________________________________________________________________________________
normalization_1 (Normalization) (None, 5)            11          input_2[0][0]                    
____

In [10]:
os.chdir(CWD)

### Initialize SurQCT parameters
SurQCT model inputs

In [11]:
surqct_inp = case_dir + '/data_gen/InputData.py'
sys.path.append(os.path.dirname(surqct_inp))

import InputData

In [20]:
# %matplotlib inline 
JqnVec_toWrite=np.linspace(0, 1, num=1, endpoint=False,dtype=int)
JqnVec = DiatData[0]['jqn'].to_numpy(int)

# iIdxVec             = np.array([4580, 4581], dtype=int)
# iIdxVec             = np.array([1,3], dtype=int)
iIdxVec=np.empty(0,dtype=int)

for i in range(np.size(JqnVec_toWrite)):    
    iDxVecTemp = [idx+1 for idx in np.arange(NLevels[0]) if (JqnVec[idx] == JqnVec_toWrite[i])]
    iIdxVec=np.append(iIdxVec,iDxVecTemp)

In [23]:
# Modifiy test input parameters
InputData.TTranVecTest   = [1500.0]
InputData.iLevelsVecTest = [1, 10, 32, 34, 35, 81, 82, 98, 1000, 2000, 3000, 4581] + list(iIdxVec)

### Generating data
Import `RatesType` module

In [24]:
RatesType = __import__(
    f'surqct.src.RatesType.{InputData.NbAtoms}Atoms.{InputData.RatesType}',
    globals(),
    locals(),
    [InputData.RatesType],
    0
)

Generate testing data

In [25]:
x_true, y_true = RatesType.generate_testingdata(InputData)
data_true = pd.concat([x_true, y_true], axis=1)

[SurQCT]:  Generating Labeled Test Data Points
[SurQCT]:    Reading Variables:  ['log_EVib', 'log_ERot', 'log_VMax', 'log_ro']
[SurQCT]:    Reading Molecule O2_UMN
[SurQCT]:    Reading Molecular Levels Data from: /home/zanardi/WORKSPACE//AirDatabase/Run0D/database/levels/modified//O2_UMN/VMin_VMax.csv
[SurQCT]:    Reading Diatomic Potential Data from: /home/zanardi/WORKSPACE//NeqPlasma/CoarseAIR/coarseair//dtb/Molecules//O2/UMN/FromUMN_Sorted.inp
[SurQCT]:    Reading Molecule O2_UMN
[SurQCT]:    Reading Molecular Levels Data from: /home/zanardi/WORKSPACE//AirDatabase/Run0D/database/levels/modified//O2_UMN/VMin_VMax.csv
[SurQCT]:    Reading Diatomic Potential Data from: /home/zanardi/WORKSPACE//NeqPlasma/CoarseAIR/coarseair//dtb/Molecules//O2/UMN/FromUMN_Sorted.inp
[SurQCT]:    Vector of Initial Levels:
[SurQCT]:    [1, 10, 32, 34, 35, 81, 82, 98, 1000, 2000, 3000, 4581, 1, 34, 81, 138, 203, 275, 355, 440, 531, 626, 728, 833, 943, 1057, 1173, 1294, 1419, 1545, 1678, 1812, 1943, 2083, 22

100%|██████████| 57/57 [00:05<00:00, 10.82it/s]


Generate prediction data

In [26]:
x_pred, y_pred = RatesType.generate_predictiondata(InputData, model.net, Labeled=True)
data_pred = pd.concat([x_pred, y_pred], axis=1)

  0%|          | 0/57 [00:00<?, ?it/s]

[SurQCT]:  Generating Prediction Data
[SurQCT]:    Reading Variables:  ['log_EVib', 'log_ERot', 'log_VMax', 'log_ro']
[SurQCT]:    Reading Molecule O2_UMN
[SurQCT]:    Reading Molecular Levels Data from: /home/zanardi/WORKSPACE//AirDatabase/Run0D/database/levels/modified//O2_UMN/VMin_VMax.csv
[SurQCT]:    Reading Diatomic Potential Data from: /home/zanardi/WORKSPACE//NeqPlasma/CoarseAIR/coarseair//dtb/Molecules//O2/UMN/FromUMN_Sorted.inp
[SurQCT]:    Reading Molecule O2_UMN
[SurQCT]:    Reading Molecular Levels Data from: /home/zanardi/WORKSPACE//AirDatabase/Run0D/database/levels/modified//O2_UMN/VMin_VMax.csv
[SurQCT]:    Reading Diatomic Potential Data from: /home/zanardi/WORKSPACE//NeqPlasma/CoarseAIR/coarseair//dtb/Molecules//O2/UMN/FromUMN_Sorted.inp
[SurQCT]:    Vector of Initial Levels:
[SurQCT]:    [1, 10, 32, 34, 35, 81, 82, 98, 1000, 2000, 3000, 4581, 1, 34, 81, 138, 203, 275, 355, 440, 531, 626, 728, 833, 943, 1057, 1173, 1294, 1419, 1545, 1678, 1812, 1943, 2083, 2221, 2365,

100%|██████████| 57/57 [00:15<00:00,  3.73it/s]


### Plotting rates
Read levels data

In [27]:
LevelsData, DiatData, NLevels = RatesType.read_levelsinfo(InputData, InputData.TTranVecTest)

[SurQCT]:    Reading Variables:  ['log_EVib', 'log_ERot', 'log_VMax', 'log_ro']
[SurQCT]:    Reading Molecule O2_UMN
[SurQCT]:    Reading Molecular Levels Data from: /home/zanardi/WORKSPACE//AirDatabase/Run0D/database/levels/modified//O2_UMN/VMin_VMax.csv
[SurQCT]:    Reading Diatomic Potential Data from: /home/zanardi/WORKSPACE//NeqPlasma/CoarseAIR/coarseair//dtb/Molecules//O2/UMN/FromUMN_Sorted.inp
[SurQCT]:    Reading Molecule O2_UMN
[SurQCT]:    Reading Molecular Levels Data from: /home/zanardi/WORKSPACE//AirDatabase/Run0D/database/levels/modified//O2_UMN/VMin_VMax.csv
[SurQCT]:    Reading Diatomic Potential Data from: /home/zanardi/WORKSPACE//NeqPlasma/CoarseAIR/coarseair//dtb/Molecules//O2/UMN/FromUMN_Sorted.inp


In [28]:
paths = {k: out_dir + f'/{k}/' for k in ('2D','3D')}
for path in paths.values():
    if not os.path.exists(path):
        os.makedirs(path)

for T in InputData.TTranVecTest:
    for iIdx in InputData.iLevelsVecTest:
        # True values
        TrueData = data_true.loc[(data_true['TTran_i'] == T) & (data_true['Idx_i'] == iIdx-1) & (data_true['KExcit'] >= 1.e-20)]
        TrueData = (TrueData['Idx_j'].to_numpy().astype(int), TrueData['KExcit'].to_numpy())
        # Pred values
        PredData = data_pred.loc[(data_pred['TTran_i'] == T) & (data_pred['Idx_i'] == iIdx-1) & (data_pred['KExcit'] >= 1.e-20)]
        PredData = (PredData['Idx_j'].to_numpy().astype(int), PredData['KExcit'].to_numpy())
        # Plotting
        fig      = RatesType.plot_prediction3D(paths['3D'], iIdx, DiatData, TrueData, PredData)
#         plt.show()
#         plt.close()
#         fig      = RatesType.plot_prediction2D(paths['2D'], iIdx, DiatData, TrueData, PredData)
#         plt.show()
#         plt.close()
            

/home/zanardi/WORKSPACE/ML/PyCOMET/pycomet/examples/SurQCT/surqct/src/RatesType/3Atoms/KExcit.py:681: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  ax = fig.add_subplot(projection='3d')


### Read Levels Data

In [ ]:
from Reading  import read_levelsdata, read_diatdata

In [ ]:
OtherVar           = InputData.OtherVar
xVarsVec_i         = InputData.xVarsVec_i 
xVarsVec_Delta     = InputData.xVarsVec_Delta
xVarsVec           = list(set(xVarsVec_i) | set(xVarsVec_Delta))

xbranch = [(VarName + '_i') for VarName in xVarsVec_i]
Tbranch = ['TTran_i']
xtrunk = [(VarName + '_Delta') for VarName in xVarsVec_Delta]
Ttrunk = ['TTran_Delta']
xTrainingVars = xbranch + Tbranch + xtrunk + Ttrunk

xVarsVec_i         = InputData.xVarsVec_i + ['vqn','jqn','EVib','ERot']
xVarsVec_Delta     = InputData.xVarsVec_Delta
xVarsVec           = list(set(xVarsVec_i) | set(xVarsVec_Delta))

NMolecules         = len(InputData.PathToDiatFile)
LevelsData = []
DiatData   = []
NLevels    = []

for iMol in range(NMolecules):
    
    LevelsDataTemp = read_levelsdata(InputData.PathToLevelsFile[iMol], xVarsVec, '')
    LevelsData.append(LevelsDataTemp)

    DiatDataTemp = read_diatdata(InputData.PathToDiatFile[iMol], InputData.Molecules[iMol], np.array([TTran]), np.array([TTran]))
    DiatData.append(DiatDataTemp)
    
    NLevelsTemp    = DiatDataTemp.shape[0]
    NLevels.append(NLevelsTemp)

### Read HDF5 Data

In [ ]:
# Reading Excitation Rates Data 
def read_kexcitdata(PathToHDF5File, TTra, TInt, NProcTypes):
    print('[SurQCT]:       Reading HDF5 File from: ' + PathToHDF5File + ' for Excitation Rates at Temperature ' + str(int(TTra)) + 'K')

    HDF5Exist_Flg = os.path.exists(PathToHDF5File)
    if (HDF5Exist_Flg):
        f = h5py.File(PathToHDF5File, 'a')
    else:
        f = {'key': 'value'}

    TStr = 'T_' + str(int(TTra)) + '_' + str(int(TInt)) + '/Rates/'
    grp  = f[TStr]

    Data  = grp["Inel"]
    KInel = Data[...]

    KOther = []
    for iProc in range(2, NProcTypes):
        ExchStr    = "Exch_" + str(iProc-1)
        Data       = grp[ExchStr] 
        KOtherTemp = Data[...]
        KOther.append(KOtherTemp)

    f.close()   


    return KInel, KOther


KInelMat, KExchMatList = read_kexcitdata(InputData.PathToHDF5File, TTran, TTran, NProcTypes)

if (ExcitType == 'KInel'):
    KExcit = KInelMat
elif (ExcitType == 'KExch'):
    KExcit = KExchMatList[0]
else:
    KExcit = KInelMat + KExchMatList[0]

# Postprocessing - Model analysis

In [ ]:
import matplotlib.ticker as mticker
# My axis should display 10⁻¹ but you can switch to e-notation 1.00e+01
def log_tick_formatter(val, pos=None):
    return f"$10^{{{int(val)}}}$"  # remove int() if you don't use MaxNLocator
    # return f"{10**val:.2e}"      # e-Notation


In [ ]:
# iIdxVec             = np.array([1, 3, 10, 32, 34, 35, 81, 82, 98, 1000, 2000, 3000, 4581], dtype=int)
# iIdxVec             = np.array([1,3], dtype=int)

### Loop on Initial States
Str = 'q_'+str(int(TTran))
for iIdx in tqdm(iIdxVec, desc='[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes'):
    iIdx -= 1
    print('Level: '+str(iIdx)+'; vqn: '+str(DiatData[0]['vqn'].to_numpy()[iIdx])+'; jqn:'+str(DiatData[0]['jqn'].to_numpy()[iIdx]))
    time.sleep(0.02)
    
    if (InputData.ExoEndoFlg):
        jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] <= DiatData[0]['EInt'].to_numpy()[iIdx])]
        jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
        RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
    else:
        jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    >= DiatData[0][Str].to_numpy()[iIdx])]
        jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
        RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
    jNLevels              = len(jIdxVec)
    jNLevelsNo            = len(jIdxVecNo)  
    
    ### FWD Rates
    iiIdxVec              = [iIdx]*jNLevels
    
    TTranVec              = np.ones((jNLevels))*TTran
    TTranDataTemp         = pd.DataFrame({'TTran': TTranVec})
    TTranDataTemp.index   = jIdxVec

    
    iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
    iLevelsDataTemp.index = jIdxVec

    jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
    if (OtherVar == '_Delta'):
        jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
    else:
        jLevelsDataTemp   = jLevelsDataTemp
    jLevelsDataTemp.index = jIdxVec
    
    kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
    kLevelsDataTemp.index = jIdxVec

    
    iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
    iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]
    
    jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
    jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

    kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
    kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]
    
    xTemp_FWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)
    
    x1 = iLevelsData[xTrainingVars[0:5]].to_numpy()
    x2 = jLevelsData[xTrainingVars[5:10]].to_numpy()    

    x = [x1,x2]
        
    if (len(xTemp_FWD[xTrainingVars]) > 0):
        KExcit_NN_FWD          = np.exp( model.net.predict(x) ) / InputData.MultFact  
    
    ## BWD Rates
    iiIdxVec              = jIdxVecNo
    jIdxVec               = [iIdx]*jNLevelsNo
    
    TTranVec              = np.ones((jNLevelsNo))*TTran
    TTranDataTemp         = pd.DataFrame({'TTran': TTranVec})
    TTranDataTemp.index   = iiIdxVec

    
    iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
    iLevelsDataTemp.index = iiIdxVec

    jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy() 
    jLevelsDataTemp.index = iiIdxVec
    if (OtherVar == '_Delta'):
        jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
    else:
        jLevelsDataTemp   = jLevelsDataTemp
    
    kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
    kLevelsDataTemp.index = iiIdxVec

    
    iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
    iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]
    
    jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
    jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

    kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
    kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]
    
    xTempp_BWD            = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)
    
    x1 = iLevelsData[xTrainingVars[0:5]].to_numpy()
    x2 = jLevelsData[xTrainingVars[5:10]].to_numpy()    
    x = [x1,x2]
    
    if (len(xTempp_BWD[xTrainingVars]) > 0):
        KExcit_NN_BWD          = np.exp( model.net.predict(x) ) / InputData.MultFact 
        KExcit_NN_BWD[:,0]     = KExcit_NN_BWD[:,0] / RatioNo

        iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        iLevelsData.columns   = [(VarName + '_j') for VarName in iLevelsData.columns]

        jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
        jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

        kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        kLevelsData.columns   = [(VarName + '_i') for VarName in kLevelsData.columns]

        xTemp_BWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)
        
    ### Concatenating
        if (len(xTemp_FWD[xTrainingVars]) > 0):
            xTemp     = pd.concat([xTemp_FWD, xTemp_BWD], axis=0)
            KExcit_NN = np.concatenate((KExcit_NN_FWD, KExcit_NN_BWD), axis=0)
        else:
            xTemp     = xTemp_BWD
            KExcit_NN = KExcit_NN_BWD
    else:
        if (len(xTemp_FWD[xTrainingVars]) > 0):
            xTemp     = xTemp_FWD
            KExcit_NN = KExcit_NN_FWD   
    
    ### Plotting
    Mask     = KExcit[iIdx,:] > 1e-15
    
#     j_color = xTemp['jqn_j'] % 2
#     v_color = DiatData[1]['vqn'][Mask] == 1
#     cmap=mpl.colors.ListedColormap(["navy","crimson","limegreen","gold","k","r","b","y","c"])
#     cmap1=mpl.colors.ListedColormap(["k","limegreen"])

    v = DiatData[0]['vqn'].to_numpy()[iIdx]
    J = DiatData[0]['jqn'].to_numpy()[iIdx]
    title = f"Level {iIdx}: (v, J) = ({v}, {J})"
    
    fig = plt.figure(figsize=(12,8))
    label_plot = f"Level: (v,J) = ({DiatData[0]['vqn'].to_numpy()[iIdx]},{DiatData[0]['jqn'].to_numpy()[iIdx]})"
    plt.scatter(DiatData[1]['jqn'][Mask], KExcit[iIdx,Mask], c='k', s=30,label='QCT')
    plt.scatter(xTemp['jqn_j'], KExcit_NN[:,0], s=20, c='r', label='surQCT')
    plt.yscale('log')
    plt.legend(['QCT','SurQCT'])
    plt.ylim([1.e-16, 1.e-8])
    plt.xlabel('J')
    plt.ylabel(r'$k_{ij}$')
    plt.title(title)
#     plt.show()
    plt.savefig(output + f'/2D_level{iIdx}_v{v}_J{J}.png', facecolor='white', transparent=False)
#     pickle.dump(fig, open(output + f'/level{iIdx}_v{v}_J{J}.fig.pickle', 'wb'))
    plt.close()
    
#     fig = plt.figure(figsize=(12,8))
#     label_plot = "iLevel - vqn:" + str(DiatData[0]['vqn'].to_numpy()[iIdx])+"; jqn:"+str(DiatData[0]['jqn'].to_numpy()[iIdx])
#     plt.scatter(DiatData[1]['jqn'][Mask], KExcit[iIdx,Mask], c='k', s=30,label='QCT')
#     plt.scatter(xTemp['jqn_j'], KExcit_NN[:,0], s=20, c='r', label='SurQCT')
#     plt.yscale('log')
#     plt.legend(['QCT','SurQCT'])
#     plt.ylim([1.e-16, 1.e-8])
#     plt.xlabel('J')
#     plt.ylabel('k(iLevel,j)')
#     plt.title(label_plot)


    fig = plt.figure(figsize=(12,8))
    ax = fig.add_subplot(projection='3d')
    ax.set_xlabel('J')
    ax.set_ylabel('v')
    ax.set_zlabel(r'$k_{ij}$')
    v = DiatData[0]['vqn'].to_numpy()[iIdx]
    J = DiatData[0]['jqn'].to_numpy()[iIdx]
    ax.set_title(f"Level {iIdx}: (v, J) = ({v}, {J})")
    ax.zaxis.set_major_formatter(mticker.FuncFormatter(log_tick_formatter))
    ax.zaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.scatter(DiatData[1]['jqn'][Mask], DiatData[1]['vqn'][Mask], np.log10(KExcit[iIdx,Mask]), c='k', s=1,label='QCT')
    ax.scatter(xTemp['jqn_j'], xTemp['vqn_j'], np.log10(KExcit_NN[:,0]), c='r', s=1, label='SurQCT')
#     plt.show()
    plt.savefig(output + f'/3D_level{iIdx}_v{v}_J{J}.png')
#     pickle.dump(fig, open(output + f'/level{iIdx}_v{v}_J{J}.fig.pickle', 'wb'))
    plt.close()
    


In [ ]:
# %matplotlib inline 
JqnVec_toWrite=np.linspace(0, 1, num=1, endpoint=False,dtype=int)
JqnVec = DiatData[0]['jqn'].to_numpy(int)

# iIdxVec             = np.array([4580, 4581], dtype=int)
# iIdxVec             = np.array([1,3], dtype=int)
iIdxVec=np.empty(0,dtype=int)

for i in range(np.size(JqnVec_toWrite)):    
    iDxVecTemp = [idx+1 for idx in np.arange(NLevels[0]) if (JqnVec[idx] == JqnVec_toWrite[i])]
    iIdxVec=np.append(iIdxVec,iDxVecTemp)


### Loop on Initial States
Str = 'q_'+str(int(TTran))
for iIdx in tqdm(iIdxVec, desc='[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes'):
    iIdx -= 1
    print('Level: '+str(iIdx)+'; vqn: '+str(DiatData[0]['vqn'].to_numpy()[iIdx])+'; jqn:'+str(DiatData[0]['jqn'].to_numpy()[iIdx]))
    time.sleep(0.02)
    
    if (InputData.ExoEndoFlg):
        jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] <= DiatData[0]['EInt'].to_numpy()[iIdx])]
        jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
        RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
    else:
        jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    >= DiatData[0][Str].to_numpy()[iIdx])]
        jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
        RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
    jNLevels              = len(jIdxVec)
    jNLevelsNo            = len(jIdxVecNo)  
    
    ### FWD Rates
    iiIdxVec              = [iIdx]*jNLevels
    
    TTranVec              = np.ones((jNLevels))*TTran
    TTranDataTemp         = pd.DataFrame({'TTran': TTranVec})
    TTranDataTemp.index   = jIdxVec

    
    iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
    iLevelsDataTemp.index = jIdxVec

    jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
    if (OtherVar == '_Delta'):
        jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
    else:
        jLevelsDataTemp   = jLevelsDataTemp
    jLevelsDataTemp.index = jIdxVec
    
    kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
    kLevelsDataTemp.index = jIdxVec

    
    iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
    iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]
    
    jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
    jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

    kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
    kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]
    
    xTemp_FWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

    
    x1 = iLevelsData[xTrainingVars[0:5]].to_numpy()
    x2 = jLevelsData[xTrainingVars[5:10]].to_numpy()    

    x = [x1,x2]
        
    if (len(xTemp_FWD[xTrainingVars]) > 0):
        KExcit_NN_FWD          = np.exp( model.net.predict(x) ) / InputData.MultFact  
    
    ## BWD Rates
    iiIdxVec              = jIdxVecNo
    jIdxVec               = [iIdx]*jNLevelsNo
    
    TTranVec              = np.ones((jNLevelsNo))*TTran
    TTranDataTemp         = pd.DataFrame({'TTran': TTranVec})
    TTranDataTemp.index   = iiIdxVec

    
    iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
    iLevelsDataTemp.index = iiIdxVec

    jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy() 
    jLevelsDataTemp.index = iiIdxVec
    if (OtherVar == '_Delta'):
        jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
    else:
        jLevelsDataTemp   = jLevelsDataTemp
    
    kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
    kLevelsDataTemp.index = iiIdxVec

    
    iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
    iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]
    
    jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
    jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

    kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
    kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]
    
    xTempp_BWD            = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)
    
    x1 = iLevelsData[xTrainingVars[0:5]].to_numpy()
    x2 = jLevelsData[xTrainingVars[5:10]].to_numpy()    
    x = [x1,x2]
    
    if (len(xTempp_BWD[xTrainingVars]) > 0):
        KExcit_NN_BWD          = np.exp( model.net.predict(x) ) / InputData.MultFact 
        KExcit_NN_BWD[:,0]     = KExcit_NN_BWD[:,0] / RatioNo

        iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        iLevelsData.columns   = [(VarName + '_j') for VarName in iLevelsData.columns]

        jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
        jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

        kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        kLevelsData.columns   = [(VarName + '_i') for VarName in kLevelsData.columns]

        xTemp_BWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)
        
    ### Concatenating
        if (len(xTemp_FWD[xTrainingVars]) > 0):
            xTemp     = pd.concat([xTemp_FWD, xTemp_BWD], axis=0)
            KExcit_NN = np.concatenate((KExcit_NN_FWD, KExcit_NN_BWD), axis=0)
        else:
            xTemp     = xTemp_BWD
            KExcit_NN = KExcit_NN_BWD
    else:
        if (len(xTemp_FWD[xTrainingVars]) > 0):
            xTemp     = xTemp_FWD
            KExcit_NN = KExcit_NN_FWD   
    
    ### Plotting
    Mask     = KExcit[iIdx,:] > 1e-15
    
#     j_color = xTemp['jqn_j'] % 2
#     v_color = DiatData[1]['vqn'][Mask] == 1
#     cmap=mpl.colors.ListedColormap(["navy","crimson","limegreen","gold","k","r","b","y","c"])
#     cmap1=mpl.colors.ListedColormap(["k","limegreen"])

    v = DiatData[0]['vqn'].to_numpy()[iIdx]
    J = DiatData[0]['jqn'].to_numpy()[iIdx]
    title = f"Level {iIdx}: (v, J) = ({v}, {J})"
    
    fig = plt.figure(figsize=(12,8))
    label_plot = f"Level: (v,J) = ({DiatData[0]['vqn'].to_numpy()[iIdx]},{DiatData[0]['jqn'].to_numpy()[iIdx]})"
    plt.scatter(DiatData[1]['jqn'][Mask], KExcit[iIdx,Mask], c='k', s=30,label='QCT')
    plt.scatter(xTemp['jqn_j'], KExcit_NN[:,0], s=20, c='r', label='surQCT')
    plt.yscale('log')
    plt.legend(['QCT','SurQCT'])
    plt.ylim([1.e-16, 1.e-8])
    plt.xlabel('J')
    plt.ylabel(r'$k_{ij}$')
    plt.title(title)
#     plt.show()
    plt.savefig(output + f'/2D_level{iIdx}_v{v}_J{J}.png')
#     pickle.dump(fig, open(output + f'/level{iIdx}_v{v}_J{J}.fig.pickle', 'wb'))
    plt.close()
    
#     fig = plt.figure(figsize=(12,8))
#     ax = fig.add_subplot(projection='3d')
#     ax.set_xlabel('J')
#     ax.set_ylabel('v')
#     ax.set_zlabel(r'$k_{ij}$')
#     ax.zaxis.set_major_formatter(mticker.FuncFormatter(log_tick_formatter))
#     ax.zaxis.set_major_locator(mticker.MaxNLocator(integer=True))
#     ax.scatter(DiatData[1]['jqn'][Mask], DiatData[1]['vqn'][Mask], np.log10(KExcit[iIdx,Mask]), c='k', s=1,label='QCT')
#     ax.scatter(xTemp['jqn_j'], xTemp['vqn_j'], np.log10(KExcit_NN[:,0]), c='r', s=1, label='SurQCT')
#     plt.show()


    fig = plt.figure(figsize=(12,8))
    ax = fig.add_subplot(projection='3d')
    ax.set_xlabel('J')
    ax.set_ylabel('v')
    ax.set_zlabel(r'$k_{ij}$')
    ax.set_title(title)
    ax.zaxis.set_major_formatter(mticker.FuncFormatter(log_tick_formatter))
    ax.zaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.scatter(DiatData[1]['jqn'][Mask], DiatData[1]['vqn'][Mask], np.log10(KExcit[iIdx,Mask]), c='k', s=1,label='QCT')
    ax.scatter(xTemp['jqn_j'], xTemp['vqn_j'], np.log10(KExcit_NN[:,0]), c='r', s=1, label='SurQCT')
#     plt.show()
    plt.savefig(output + f'/3D_level{iIdx}_v{v}_J{J}.png', facecolor='white', transparent=False)
#     pickle.dump(fig, open(output + f'/level{iIdx}_v{v}_J{J}.fig.pickle', 'wb'))
    plt.close()
    